# 📦 Análisis de Eficiencia de Carga: Geometría de Pallets

Este notebook valida la lógica de cubicaje del optimizador, evaluando cuántos pallets estándar (EUR/EPAL) caben físicamente en un semirremolque de 13.6 metros bajo diferentes configuraciones.

### 📐 Base Matemática del Cubicaje

La capacidad máxima de unidades de carga en el suelo del camión se determina mediante la optimización de la orientación de los pallets (Longitudinal vs. Transversal):

$$ C_{suelo} = \max \left( \lfloor \frac{L_{camion}}{l_{pal}} \rfloor \times \lfloor \frac{W_{camion}}{w_{pal}} \rfloor, \quad \lfloor \frac{L_{camion}}{w_{pal}} \rfloor \times \lfloor \frac{W_{camion}}{l_{pal}} \rfloor \right) $$

Donde:
- $L_{camion}, W_{camion}$: Largo (13.6m) y Ancho (2.4m) del contenedor.
- $l_{pal}, w_{pal}$: Largo (1.2m) y Ancho (0.8m) del pallet EUR.

### 🏗️ Apilamiento Vertical (Capacidad 3D)

Si la carga es apilable ($stackable = True$):

$$ C_{total} = C_{suelo} \times \lfloor \frac{H_{camion}}{h_{pal}} \rfloor $$

Donde $H_{camion}$ es la altura útil y $h_{pal}$ la altura del bulto.

In [1]:
import sys
import os
import pandas as pd

# Añadir el directorio raíz al path para importar logistic_core
sys.path.append(os.path.abspath(os.path.join('..', '..', '..')))

from logistic_core.utils.capacity_estimator import TruckCapacityEstimator, Pallet

# Inicialización del estimador con dimensiones estándar europeas
estimator = TruckCapacityEstimator(
    container_length_m=13.6,
    container_width_m=2.4,
    container_height_m=2.7
)

eur_pallet = Pallet(length_m=1.2, width_m=0.8, height_m=1.3) # Pallet de media altura

## 1. Validación de Capacidad Estándar

Comparamos la capacidad entre carga no apilable y apilable.

In [2]:
cap_single = estimator.capacity(eur_pallet, stackable=False)
cap_stack = estimator.capacity(eur_pallet, stackable=True)

print(f"Configuración No Apilable: {cap_single['summary']}")
print(f"Configuración Apilable: {cap_stack['summary']}")

Configuración No Apilable: 17 columnas × 2 filas × 1 capas = 34 pallets
Configuración Apilable: 17 columnas × 2 filas × 2 capas = 68 pallets


## 2. Prueba Forense de Espacio

Verificamos si un pedido específico de 40 pallets de media altura (1.3m) cabe en el vehículo.

In [3]:
pedido_n = 40
check = estimator.fits(n_pallets=pedido_n, pallet=eur_pallet, stackable=True)
print(f"Resultado Pedido {pedido_n} pallets: {check['summary']}")

Resultado Pedido 40 pallets: ✓ Cabe: 40 solicitados, 68 disponibles (excedente 28)


### Conclusión Técnica
El modelo confirma que la capacidad máxima en suelo para pallets EUR orientados transversalmente es de **33 unidades**. El uso de algoritmos de optimización de orientación permite aprovechar al máximo los 2.4 metros de ancho del remolque, evitando la pérdida de slots que ocurriría con una orientación fija ineficiente.